In [1]:
import torch
import torch.nn as nn
import numpy as np

In [9]:
class FeedForward(nn.Module):
    def __init__(self,d_model,d_ff):
        super().__init__()
        self.fc1=nn.Linear(d_model,d_ff)
        self.activation =nn.ReLU()
        self.fc2=nn.Linear(d_ff,d_model)

    def forward(self,x):
        x=self.fc1(x)
        x=self.activation(x)
        x=self.fc2(x)
        return x

In [4]:
class GQA(nn.Module):
    def __init__(self, d_model, num_heads, num_kv_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.d_k = d_model // num_heads
        self.group_size = num_heads // num_kv_heads

        self.W_q = nn.Linear(d_model,self.d_k*num_heads)
        self.W_k = nn.Linear(d_model,self.d_k*num_kv_heads)
        self.W_v = nn.Linear(d_model,self.d_k*num_kv_heads)
        self.W_o = nn.Linear(d_model,d_model)

    def forward(self, x):
        seq_len = x.shape[0]

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

       
        Q = Q.view(seq_len, self.num_heads, self.d_k).transpose(0,1)
        K = K.view(seq_len, self.num_kv_heads, self.d_k).transpose(0,1)
        V = V.view(seq_len, self.num_kv_heads, self.d_k).transpose(0,1)


       
        K = torch.repeat_interleave(K, self.group_size, dim=0)
        V = torch.repeat_interleave(V, self.group_size, dim=0)

        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        scores = Q @ K.transpose(-2, -1) / (self.d_k ** 0.5)
        scores = scores.masked_fill(mask, float('-inf'))
        weights = torch.softmax(scores, dim=-1)
        attn_out = weights @ V

        output = attn_out.transpose(0,1).reshape(seq_len, self.d_model)
        output = self.W_o(output)
        return output


In [10]:
class TransfomerBlock(nn.Module):
    def __init__(self,d_model,num_heads,num_kv_heads,d_ff):
        super().__init__()
        self.attn = GQA(d_model, num_heads, num_kv_heads)
        self.ff=FeedForward(d_model,d_ff)
        self.norm1=nn.LayerNorm(d_model)
        self.norm2=nn.LayerNorm(d_model)

def forward(self, x):
    x = x + self.attn(self.norm1(x))
    x = x + self.ff(self.norm2(x))
    return x



In [11]:
block = TransfomerBlock(d_model=16, num_heads=4, num_kv_heads=2, d_ff=64)
x=torch.randn(6,16)
out=block(x)
print(x.shape)

NotImplementedError: Module [TransfomerBlock] is missing the required "forward" function